# Temporally naive layer-attention walkthrough

This notebook uses dummy images and a tiny frozen transformer-like network. Nothing is trained. It exposes each operation performed by `TemporalNaiveLayerAttention`:

1. extract hooked layer features;
2. project each layer into a common value space;
3. convert time-position embeddings into layer weights;
4. combine projected layers;
5. produce latent representations and optional predictions.

The layer weights depend only on time position and are shared across images. There is no self-attention, recurrence, or temporal interaction.

In [2]:
import sys
from pathlib import Path

import torch
from torch import nn

# Find the repository root whether Jupyter starts there or in python_scripts/scripts.
root_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]
PROJECT_ROOT = next(path for path in root_candidates if (path / 'config.yaml').exists())
sys.path.insert(0, str(PROJECT_ROOT / 'python_scripts' / 'src'))

from model_classes.temporal_models import TemporalNaiveLayerAttention

torch.manual_seed(0)

## 1. A tiny frozen backbone and an imgANN-compatible wrapper

The backbone converts each image into four tokens of width 8, then applies two small token-wise layers. `TinyImgANN` registers real PyTorch forward hooks and mean-pools tokens, reproducing the part of the project `imgANN` interface used by the temporal model.

In [3]:
class TinyBackbone(nn.Module):
    def __init__(self, embedding_dim=8):
        super().__init__()
        # Four non-overlapping 4x4 patches are embedded into width E=8.
        self.patch_embedding = nn.Conv2d(3, embedding_dim, kernel_size=4, stride=4)
        # These layers preserve E, like transformer blocks in one residual width.
        self.layer_0 = nn.Sequential(nn.Linear(embedding_dim, embedding_dim), nn.GELU())
        self.layer_1 = nn.Sequential(nn.Linear(embedding_dim, embedding_dim), nn.GELU())

    def forward(self, pixel_values):
        patches = self.patch_embedding(pixel_values)          # [B, E, 2, 2]
        tokens = patches.flatten(2).transpose(1, 2)           # [B, 4, E]
        layer_0_tokens = self.layer_0(tokens)                 # [B, 4, E]
        return self.layer_1(layer_0_tokens)                   # [B, 4, E]


class TinyImgANN:
    def __init__(self):
        self.model = TinyBackbone().eval()
        self.features = {}
        self.handles = {}

    def get_model(self):
        return self.model

    def get_pkg(self):
        # The temporal model converts tensor inputs to {'pixel_values': tensor}.
        return 'hf'

    def create_forward_hook(self, layer_names):
        # Remove old hooks before registering the requested ordered layer set.
        for handle in self.handles.values():
            handle.remove()
        self.features = {}
        self.handles = {}
        modules = dict(self.model.named_modules())

        for layer_name in layer_names:
            def save_mean_tokens(module, inputs, output, name=layer_name):
                self.features[name] = output.mean(dim=1)      # [B, E]

            self.handles[layer_name] = modules[layer_name].register_forward_hook(
                save_mean_tokens
            )
        return self.features, self.handles

    def extract_features(self, x):
        # The real imgANN also freezes feature extraction with no_grad.
        with torch.no_grad():
            self.model(**x)
        return self.features

## 2. Dummy images and the temporal model

We use two 8x8 RGB images, two hooked layers, and three time bins. The transformer feature width (8) is deliberately not passed to the temporal model: each `LazyLinear` infers it during the first forward computation.

In [4]:
B, C, H, W = 2, 3, 8, 8
K, T = 2, 3
images = torch.randn(B, C, H, W)
layer_names = ['layer_0', 'layer_1']

img_ann = TinyImgANN()
model = TemporalNaiveLayerAttention(
    img_ann=img_ann,
    layer_names=layer_names,
    n_time_bins=T,
    position_embedding_dim=4,
    layer_projection_dim=5,
    latent_dim=6,
    output_dim=2,
)

print('dummy images:', images.shape)
print('registered hooks:', list(img_ann.handles))
print('feature_dim before extraction:', model.feature_dim)

dummy images: torch.Size([2, 3, 8, 8])
registered hooks: ['layer_0', 'layer_1']
feature_dim before extraction: None


## 3. Extract hooked features

The frozen backbone runs once. Each hook captures `[B, tokens, E]`, mean-pools tokens, and the temporal model stacks layers into `[B, K, E]`.

In [5]:
with torch.inference_mode():
    layer_features = model._extract_layer_features(images)

print('layer features [B, K, E]:', layer_features.shape)
print('inferred transformer feature_dim:', model.feature_dim)
print('first image, first layer:', layer_features[0, 0])

layer features [B, K, E]: torch.Size([2, 2, 8])
inferred transformer feature_dim: 8
first image, first layer: tensor([ 0.0572,  0.1637,  0.0010,  0.1120,  0.0635, -0.0132,  0.0793,  0.0592])


## 4. Align each layer in a common value space

Every layer has its own learned `E → V` projection. The parameters are random and remain untrained here. Separate projections avoid directly averaging potentially mismatched coordinates from different transformer depths.

In [6]:
with torch.inference_mode():
    projected_layers = torch.stack(
        [
            projection(layer_features[:, layer_idx])
            for layer_idx, projection in enumerate(model.layer_projections)
        ],
        dim=1,
    )

print('projected layers [B, K, V]:', projected_layers.shape)
print('first projection inferred in_features:', model.layer_projections[0][0].in_features)

projected layers [B, K, V]: torch.Size([2, 2, 5])
first projection inferred in_features: 8


## 5. Position embeddings produce layer attention

For each time index, a learned positional vector is mapped to `K` logits and normalized across layers:

$$\alpha_t = \operatorname{softmax}(W_a p_t + b_a).$$

No image features enter this calculation, so the same `[T, K]` schedule is used for all images.

In [7]:
time_idx = torch.arange(T)
position_features = model.position_embeddings[time_idx]       # [T, P]
layer_logits = model.layer_attention(position_features)       # [T, K]
time_layer_attention = torch.softmax(layer_logits, dim=-1)    # [T, K]
attention = time_layer_attention.unsqueeze(0).expand(B, -1, -1)

print('position embeddings [T, P]:', position_features.shape)
print('attention [B, T, K]:', attention.shape)
print('time x layer schedule:')
print(time_layer_attention)
print('row sums:', time_layer_attention.sum(dim=-1))
print('identical across images:', torch.equal(attention[0], attention[1]))

position embeddings [T, P]: torch.Size([3, 4])
attention [B, T, K]: torch.Size([2, 3, 2])
time x layer schedule:
tensor([[0.4975, 0.5025],
        [0.4912, 0.5088],
        [0.4990, 0.5010]], grad_fn=<SoftmaxBackward0>)
row sums: tensor([1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)
identical across images: True


## 6. Weighted layer combination, latent, and prediction

At each time position, aligned layer vectors are combined using the positional weights:

$$c_{b,t} = \sum_k \alpha_{t,k} v_{b,k}.$$

A shared MLP maps each context to the latent representation, and the optional head maps latent vectors to neural predictions.

In [8]:
with torch.inference_mode():
    context = torch.einsum('btk,bkv->btv', attention, projected_layers)
    latent_manual = model.latent_projection(context)
    pred_manual = model.prediction_head(latent_manual)

print('context [B, T, V]:', context.shape)
print('latent [B, T, latent_dim]:', latent_manual.shape)
print('prediction [B, T, output_dim]:', pred_manual.shape)

context [B, T, V]: torch.Size([2, 3, 5])
latent [B, T, latent_dim]: torch.Size([2, 3, 6])
prediction [B, T, output_dim]: torch.Size([2, 3, 2])


## 7. Verify the complete forward pass

The public `forward` method should reproduce the manually exposed steps exactly. This is still inference only—no optimizer, backward pass, or parameter update is used.

In [9]:
with torch.inference_mode():
    pred, latent, attention_forward = model(images)

torch.testing.assert_close(attention_forward, attention)
torch.testing.assert_close(latent, latent_manual)
torch.testing.assert_close(pred, pred_manual)

print('Forward equals the manual steps.')
print('pred:', pred.shape, 'latent:', latent.shape, 'attention:', attention_forward.shape)

Forward equals the manual steps.
pred: torch.Size([2, 3, 2]) latent: torch.Size([2, 3, 6]) attention: torch.Size([2, 3, 2])


## Shape summary

| Quantity | Shape | Meaning |
|---|---:|---|
| Images | `[B, C, H, W]` | Input image batch |
| Hooked features | `[B, K, E]` | Frozen transformer layer vectors |
| Projected layers | `[B, K, V]` | Layer-aligned value vectors |
| Position embeddings | `[T, P]` | Learned temporal positions |
| Attention | `[B, T, K]` | Position-only weights over layers |
| Context | `[B, T, V]` | Weighted layer combination |
| Latent | `[B, T, L]` | Shared latent representation |
| Prediction | `[B, T, O]` | Optional neural output |